# RiverWatch2 — GPU backtest dumps (Kaggle)

Seed-ensembles the freshly trained δHBV members and writes `--dump-windows` csv.gz at the **same grid** as the existing v2r dumps (stride-14/ss3), so they inner-join in `combine_dumps.py`. **Asserts the CAMELS static overlay prints** — without it NSE craters to 0.40.


In [ ]:
# --- GPU init: verify CUDA is really there BEFORE spending a session ---
import subprocess
subprocess.run(['nvidia-smi'], check=False)
import torch
assert torch.cuda.is_available(), \
    'NO CUDA GPU — set Accelerator to GPU in the notebook settings!'
print('torch', torch.__version__, '| CUDA', torch.version.cuda,
      '| device', torch.cuda.get_device_name(0),
      '| capability', torch.cuda.get_device_capability(0))

# --- Setup: clone repo (SHA-logged). Kaggle has torch/pandas/numpy already ---
import os, sys, textwrap
os.chdir('/kaggle/working')
if not os.path.exists('riverwatch2'):
    subprocess.run(['git','clone','--depth','1','--branch','benchmark-competition-2026-07',
                    'https://github.com/andrewnakas/riverwatch2.git'], check=True)
os.chdir('/kaggle/working/riverwatch2')
sha = subprocess.run(['git','rev-parse','--short','HEAD'],
                     capture_output=True, text=True).stdout.strip()
print('repo SHA:', sha)
# Kaggle already has torch/pandas/numpy/scikit-learn; nothing else is needed.
os.environ['RW2_ENABLE_MBLSTM'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'


In [ ]:
# --- Wire Kaggle Dataset inputs to repo-relative paths ---
import glob, shutil, os
# Static attrs + gauge ids + station registry (load-bearing: without
# camels_attrs.json the static overlay is all-NaN and NSE craters to 0.40).
STATIC_DS = '/kaggle/input/rw2-camels-static'
for f in ['camels_attrs.json','camels_gauge_ids.json','stations_40_enriched.json']:
    src = os.path.join(STATIC_DS, f)
    if os.path.exists(src):
        shutil.copy(src, f'data/{f}')
        print('staged', f)
    else:
        print('WARNING missing static input:', src)
# Corpora: one Dataset per forcing. Resolve the dir that holds the 531 csv.gz.
def corpus_dir(forcing):
    cands = glob.glob(f'/kaggle/input/rw2-camels-corpus-{forcing}/**/camels_corpus_{forcing}_v2',
                      recursive=True) or \
            glob.glob(f'/kaggle/input/rw2-camels-corpus-{forcing}/**/*.csv.gz', recursive=True)
    if not cands: raise FileNotFoundError(f'no corpus for {forcing}')
    d = cands[0]
    return d if os.path.isdir(d) else os.path.dirname(d)
for F in ['daymet','maurer','nldas']:
    try: print(F, '->', corpus_dir(F), len(glob.glob(corpus_dir(F)+'/*.csv.gz')), 'basins')
    except Exception as e: print(F, 'NOT MOUNTED', e)


In [ ]:
# Add the trained-ckpts Dataset as an input, then dump per forcing.
import subprocess, glob, os
FORCING = 'daymet'
CORPUS = corpus_dir(FORCING)
cks = sorted(glob.glob(f'/kaggle/input/rw2-noq-ckpts/**/camels531_{FORCING}_dhbv_combined100_s*.pt',
                       recursive=True))
assert cks, 'no trained ckpts mounted — add rw2-noq-ckpts as input'
ckpt = ':'.join(cks)
out = f'/kaggle/working/camels531_{FORCING}_combined100_ens_s14.csv.gz'
cmd = (f'python scripts/backtest_mblstm.py --ckpt {ckpt} --corpus-dir {CORPUS} '
       f'--start 1989-10-01 --end 1999-09-30 --stride 14 --stride-stations 3 '
       f'--camels-subset 531 --label {FORCING}_combined100 --dump-windows {out}')
log = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(log.stdout[-2000:]); print(log.stderr[-500:])
assert 'CAMELS static overlay' in log.stdout, 'OVERLAY MISSING — NSE will be ~0.40!'
print('OK dump ->', out)


Download the `*_combined100_ens_s14.csv.gz` from the notebook output into `data/mblstm/gpu_dumps_s14/` locally, then combine with `combine_dumps.py`.
